In [ ]:
import numpy as np
import mne
import pyxdf
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch, welch
import pandas as pd
import padasip as pa
import seaborn as sns
from scipy.stats import mannwhitneyu, shapiro, probplot, ttest_ind
from scipy import signal
from scipy import stats

fs = 500
#bands = {'Delta': (1, 4), 'Theta': (4, 8), 'Alpha': (8, 13), 'Beta': (13, 30), 'Gamma': (30, 40)}
bands = {'Alpha': (8, 13), 'Beta': (13, 30)}

In [ ]:
#Signal loading from .gdf format (use has_eog  = True for loading additional EOG channels)

def load_gdf(ruta_archivo, has_eog = False, fs = 500):

    raw = mne.io.read_raw_gdf(ruta_archivo, preload = True, verbose = False)

    if has_eog: 
         objective_channels = ['FP1', 'FP2', 'C3', 'C4', 'EOG1', 'EOG2', 'EOG3'] 
    else:
        objective_channels = ['FP1', 'FP2', 'C3', 'C4'] #C3 & C4 were the default SMARTING signal port names used for AF3 & AF4 recohtively
        
    all_channels = raw.ch_names
    
    # Isolation of objective (desired) channels
    objective_indices = []
    for channel in objective_channels:
        if channel in all_channels:
            objective_indices.append(all_channels.index(channel))
            
    # Desired rows are extracted (in µV)
    signals = raw.get_data(picks = objective_indices) * 1e6 

    #Extraction of marker events and their original indices
    events, dict_events = mne.events_from_annotations(raw, verbose = False)
    
    times_left = []
    times_right = []
    times_rest = []
    
    # Openvibe code mapping
    code_left = dict_events.get('769') 
    code_right = dict_events.get('770')
    
    #List filling
    for event in events:
        original_sample_index = event[0]
        event_id = event[2]
        
        original_time_s = original_sample_index / fs
        
        
        #For left MI events
        if event_id == code_left:
            times_left.append(round(original_time_s , 3))
            times_rest.append(round(original_time_s - 2.0, 3))
            
        #For right MI events
        elif event_id == code_right:
            times_right.append(round(original_time_s , 3))
            times_rest.append(round(original_time_s  - 2.0, 3))

    return signals, times_left, times_right, times_rest

In [1]:
#EEG Filtering 

#1 - 40 Hz Bp
def apply_bandpass(data, lowcut = 1.0, highcut = 40.0, fs = 500.0, order = 4):
    
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype = 'band')
    
    return filtfilt(b, a, data)

#50 Hz Notch
def apply_notch(data, f0 = 50.0, fs = 500.0, Q = 30.0):
    
    b, a = iirnotch(f0, Q, fs)
    return filtfilt(b, a, data)

#Whole EEG filtering pipeline
def complete_eeg_filtering(signals, fs = 500.0):
    
    filtered_signals = []
    
    for sig in signals:
        sig_centered = sig - np.mean(sig)
        sig_bp = apply_bandpass(sig_centered, fs = fs)
        sig_filt = apply_notch(sig_bp, fs = fs)
        filtered_signals.append(sig_filt)
        
    return np.array(filtered_signals)

In [3]:
#Ocular noiser removal functions

#EOG_v, and EOG_ extraction
def extract_eog_difs(signals, filter = False):
    eog = {}
    
    for tag, data_mat in signals.items():
        
        eog_h = data_mat[4] - data_mat[6] #Horizontal: Left - Right
        
        eog_v = data_mat[0] - data_mat[5] #Vertical: Top - bottom

        #Used for EEG filtering after EOG dif extraction, not applied during for final results
        if filter:
            eog_total = np.array([eog_h, eog_v])
            eog_total_filt = complete_eeg_filtering(eog_total)
            
            eog_h = eog_total_filt[0]
            eog_v = eog_total_filt[1]
        
        eog[tag] = np.column_stack((eog_h, eog_v))
        
    return eog

#Bipolar derivation
def to_bipolar(signals):
    
    bipolar1 = signals[2, :] - signals[0, :] #Left bipolar
    bipolar2 = signals[3, :] - signals[1, :] #Right bipolar
    
    return np.array([bipolar1, bipolar2])

#NLMS filtering
def nlms_filtering(signals, eog, delay = 0.25, mu = 0.1, fs = 500):
    
    filtered_signals = {}

    for tag, data_mat in signals.items():

        eog_original = eog[tag]

        #Retardation of EOG signals
        delay_samps = int(delay * fs)
        samp_num = eog_original.shape[0]
            
        delayed_eog_h = np.zeros(samp_num)
        delayed_eog_v = np.zeros(samp_num)
            
        delayed_eog_h[delay_samps:] = eog_original[:-delay_samps, 0]
        delayed_eog_v[delay_samps:] = eog_original[:-delay_samps, 1]
            
        eog_ref = np.column_stack((eog_original[:, 0], eog_original[:, 1], delayed_eog_h, delayed_eog_v))
        n_weights = 4

        clean_mat = np.zeros_like(data_mat)
        n = data_mat.shape[0]

        for i in range(n):
            
            d = data_mat[i, :]
            f = pa.filters.FilterNLMS(n = n_weights, mu = mu, w = "zeros")

            # y = Estimated ocular noise
            # e = Clean signal (diff between d and y)
            # w = Weight evoultion 
            y, e, w = f.run(d, eog_ref)

            clean_mat[i, :] = e
        filtered_signals[tag] = clean_mat
        
    return filtered_signals


#Common Average Reference filtering
def to_car(signals):
    
    signals_car = {}

    for tag, data_mat in signals.items():
        
        common_avg = np.mean(data_mat, axis = 0)
        ref_mat = data_mat - common_avg
        signals_car[tag] = ref_mat
        
    return signals_car


In [6]:
#Feature extraction functions

#Dataframe creation
def to_df(rest, left, right, param_name):   
    
    complete_data = []
    
    #Aux function
    def process_dict(param_list, condition_tag):
        for window in param_list:
            for band, param_value in window.items():
                complete_data.append({'Condition': condition_tag,'Band': band, param_name : param_value})
                
    process_dict(rest, 'Rest')
    process_dict(left, 'Left')
    process_dict(right, 'Right')
    
    return pd.DataFrame(complete_data)
    
#Relative energy computation per band
def compute_relative_energy(signals, id_rest, id_left, id_right, fs = 500):
    
    channel_names = ['Fp1', 'Fp2', 'AF3', 'AF4']

    # Trial balance measure (not applied on final results)
    min_trials = min(len(id_left), len(id_right))
    id_left, id_right = id_left[:min_trials], id_right[:min_trials]
    id_rest = id_rest[:min_trials * 2]

    def extract_re(channel, indices):
        
        out = []
        
        for start_sec in indices:
            start, end = int(start_sec * fs), int((start_sec + 2) * fs)
            if end > len(channel):
                continue

            #1Hz Welch´s PSD (50% overlap)
            f, Pxx = signal.welch(channel[start : end], fs, nperseg = int(fs))
            total = np.sum(Pxx[(f >= 1) & (f <= 40)]) + 1e-9
            window = {}
            
            for band, (f_min, f_max) in bands.items():
                idx = (f >= f_min) & (f <= f_max) if band == 'Gamma' else (f >= f_min) & (f < f_max) 
                window[band] = np.sum(Pxx[idx]) / total if np.any(idx) else 0.0
                
            out.append(window)
        return out

    dfs = {}
    
    for i, name in enumerate(channel_names):
        ch = signals[i]
        dfs[name] = to_df(extract_re(ch, id_rest), extract_re(ch, id_left), extract_re(ch, id_right), 'Relative Energy')
        
    return dfs


#Ipsilateral Coherence Computation
def ipsilateral_coherence_ratio(signals, id_rest, id_left, id_right, fs = 500):

    # Trial balance measure (not applied on final results)
    min_trials = min(len(id_left), len(id_right))
    id_left, id_right = id_left[:min_trials], id_right[:min_trials]
    id_rest = id_rest[:min_trials * 2]

    last_band = list(bands.keys())[-1]

    def band_mask(f, f_min, f_max, is_last):
        return (f >= f_min) & (f <= f_max) if is_last else (f >= f_min) & (f < f_max)

    def extract_coh(indices, subparameter):
        out = []
        for start_sec in indices:
            start, end = int(start_sec * fs), int((start_sec + 2) * fs)
            if end > len(signals[0]):
                continue
                
            v_fp1 = signals[0][start:end]
            v_af3 = signals[2][start:end]
            v_fp2 = signals[1][start:end]
            v_af4 = signals[3][start:end]

            #Left and right Coherence by cross PSD (1Hz, 50% overlap)
            
            if subparameter == 'mscoh': #MSCOH
                f, left_coh  = signal.coherence(v_fp1, v_af3, fs, nperseg = int(fs))
                f, right_coh = signal.coherence(v_fp2, v_af4, fs, nperseg = int(fs))
                
            else:  #iCOH
                f, Pxx_l = signal.csd(v_fp1, v_fp1, fs, nperseg = int(fs))
                f, Pyy_l = signal.csd(v_af3, v_af3, fs, nperseg = int(fs))
                f, Pxx_r = signal.csd(v_fp2, v_fp2, fs, nperseg = int(fs))
                f, Pyy_r = signal.csd(v_af4, v_af4, fs, nperseg = int(fs))
                f, Pxy_l = signal.csd(v_fp1, v_af3, fs, nperseg = int(fs))
                f, Pxy_r = signal.csd(v_fp2, v_af4, fs, nperseg = int(fs))
                left_coh  = np.abs(np.imag(Pxy_l / np.sqrt(Pxx_l * Pyy_l)))
                right_coh = np.abs(np.imag(Pxy_r / np.sqrt(Pxx_r * Pyy_r)))

            window = {}
            for band, (f_min, f_max) in bands.items():
                idx = band_mask(f, f_min, f_max, band == last_band)
                
                if np.any(idx):
                    window[band] = np.mean(left_coh[idx]) / (np.mean(right_coh[idx]) + 1e-9)
                    
                else:
                    window[band] = 0.0
                    
            out.append(window)
            
        return out

    mscoh_df = to_df(extract_coh(id_rest, 'mscoh'), extract_coh(id_left, 'mscoh'), extract_coh(id_right, 'mscoh'), 'Ipsilateral MSCOH ratio')
    icoh_df  = to_df(extract_coh(id_rest, 'icoh'), extract_coh(id_left, 'icoh'), extract_coh(id_right, 'icoh'), 'Ipsilateral iCOH ratio')
    
    return mscoh_df, icoh_df

In [ ]:
#Stat export to excel

def export_stats_excel(dfs_dict, file_name, parameter, subparameter, value_col = None):
    
    results = []
    
    for recording, df_data in dfs_dict.items():

        #Evaluated parameter value column
        vcol = None 
        for c in df_data.columns:
            if c not in ('Condition', 'Band'):
                vcol = c
                break

        for band in df_data['Band'].unique():
            df_banda = df_data[df_data['Band'] == band]
            rest_data = df_banda[df_banda['Condition'] == 'Rest'][vcol]
            left_data  = df_banda[df_banda['Condition'] == 'Left'][vcol]
            right_data  = df_banda[df_banda['Condition'] == 'Right'][vcol]
            total_mi  = pd.concat([left_data, right_data])
                
            normal = True
            valid_conditions = 0
            
            for condition_data in [rest_data, left_data, right_data, total_mi ]:
                if len(condition_data) >= 3:
                    valid_conditions += 1
                    if shapiro(condition_data)[1] < 0.05:
                        normal = False
                else:
                    normal = False
            if valid_conditions < 4:
                normal = False

            if normal:
                def test(g1, g2):
                    return ttest_ind(g1, g2, equal_var = False)[1] 
            else:
                def test(g1, g2):
                    return mannwhitneyu(g1, g2, alternative = 'two-sided')[1]

                        
            results.append({
                'Recording': recording,
                'Parameter': parameter,
                'Subparameter': subparameter, #Used for MSCOH / iCOH distinction
                'Band': band,
                'P-value (Rest vs. Left MI)': test(rest_data, left_data),
                'P-value (Rest vs. Right MI)': test(rest_data, right_data),
                'P-value (Rest vs. Total MI)': test(rest_data, total_mi ),
                'P-value (Left vs. Right MI)': test(left_data, right_data),
            })

    df_resultados = pd.DataFrame(results)
    df_resultados.to_excel(file_name, index = False)

In [ ]:
#Runnable example

# 1. Signal load
signals, times_left, times_right, times_rest = load_gdf('Recordings/example.gdf', has_eog=False)

# 2. EEG filtering (no ocular filtering strat)
filtered = complete_eeg_filtering(signals)

# 3. Feature extraction
re_dfs = compute_relative_energy(filtered, times_rest, times_left, times_right)  
mscoh_df, icoh_df = ipsilateral_coherence_ratio(filtered, times_rest, times_left, times_right)

# 4. Excel export
rec = 'example_rec'
re_dfs_named = {f"{rec}_{channel}": df for channel, df in re_dfs.items()}

export_stats_excel(re_dfs,           'example_RE.xlsx',    'Relative energy', 'None')
export_stats_excel({rec: mscoh_df},  'example_MSCOH.xlsx', 'Ipsilateral coherence ratio', 'MSCOH')
export_stats_excel({rec: icoh_df},   'example_iCOH.xlsx',  'Ipsilateral coherence ratio', 'iCOH')